# Step 4 extension: two more embedding settings for ESIM
**Why:** with word2vec `min_count=2`, every rare house number becomes the same `UNK`, so a same-street case1 pair with two rare numbers is the same input as an identical pair with a rare number (02, D1–D4). fastText fixed that but cost ~10 pts on case2s (02, D5). This notebook tests two settings that target the numbers without (option 5) or while also (option 4) changing how street words are represented.

**Code:** written by Claude (2026-11-06, at David's request; David does the results analysis). The model is the same ESIM as 02, imported from `esim_model.py` with the table and sizes as arguments; the tables come from `esim_diagnostics.py`. Section 1 checks that the module reproduces 02's saved predictions.

| | option 5: random rows | option 4: stacked |
|---|---|---|
| setting name | `rand_mc2` | `stack_mc1` |
| source vectors | word2vec `mc2` | word2vec `mc1` and fastText `mc1` |
| vocab | `PAD` 0, `UNK` 1, word2vec words (+2), then every token word2vec dropped | `PAD` 0, `UNK` 1, every word2vec `mc1` word (+2) |
| matrix rows | 2 zero rows + word2vec rows + one random row per dropped token | 2 zero rows + `[w2v.wv[w]; ft.wv[w]]` per word |
| random rows | normal, per-dimension mean / std of the word2vec rows, seed 0, frozen, same for all training seeds | — |
| embedding size | 200 | 400 (context BiLSTM input 400, hidden stays 200) |
| run folder | `checkpoints/esim_rand_mc2/` | `checkpoints/esim_stack_mc1/` |
| protocol | as 02: 3 seeds, `is_better` (case1 guardrail 98), patience 3, val/test predictions saved | same |

**Expectation:** option 5 gives `3015` its own fixed vector (same on both sides, unrelated to `3017`) without changing street words → numbers fixed, case2s near `mc2`. Option 4 hands the model both signals; the fastText half brings look-alike streets close again, so case2s depends on whether training learns to ignore it.

## 0. Setup

In [2]:
%load_ext autoreload
%autoreload 2
import os
import sys
sys.path.append("..")   # geocoding_ml/

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

from helpers.pairs import load_pairs
from siamese_model.data_plumbing import PairDataset
from siamese_model.training_step import predict, train_epoch
from siamese_model.real_training import broken_run_problems, THRESHOLD
from esim.tokenizer import tokenize
from esim.esim_model import (ESIM, UNK_ID, plain_table, random_rows_table, stacked_table,
                             encode_batch, make_collate, make_loaders, run_training, save_run, BATCH_SIZE)
from esim.esim_diagnostics import SETTINGS, SEEDS, health, slice_table, rare_number_group, rare_number_table

DATASET = "overture_case2s"
pd.set_option("display.width", 250)
pd.set_option("display.expand_frame_repr", False)

train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
splits = {"train": (train, train_label), "val": (val, val_label), "test": (test, test_label)}
all_addr = pd.concat([d[c] for d in (train, val, test) for c in ("address_a", "address_b")])
print(len(train), len(val), len(test))

202568 41865 43844


## 1. Equivalence check: the module ESIM = 02's ESIM
- Claude made some changes to model, ensuring no changes affected the preds
- Test current model weights with previous output

In [3]:
vocab_mc2, matrix_mc2 = plain_table("word2vec", 2)
ckpt = torch.load("checkpoints/esim_mc2/esim_seed_0.pt")
assert ckpt["vocab"] == vocab_mc2, "vocab differs from the one 02 trained with"

model = ESIM(matrix_mc2)
model.load_state_dict(ckpt["state_dict"])
val_loader_mc2 = make_loaders(vocab_mc2, {"val": splits["val"]})["val"]
preds = (predict(model, val_loader_mc2) >= THRESHOLD).int().numpy()
saved = np.load("checkpoints/esim_mc2/preds_val_seed_0.npy")
assert np.array_equal(preds, saved), f"{(preds != saved).sum()} predictions differ from 02"
print("module ESIM reproduces 02 (mc2, seed 0, val):", len(preds), "predictions identical")

module ESIM reproduces 02 (mc2, seed 0, val): 41865 predictions identical


### 2. Option 5: word2vec `mc2` + one fixed random row per dropped token
- `random_rows_table` creates the data needed for running random row. 
- We do so by figuring out which values are included in word2vec(mc=2) vs in vocab, and then creating a corresponding vocab and matrix.
    - vocab means what `word -> id`
    - matrix mean what we use for the embedding table
- **NOTE** We create our random values by using the std dev of our wv.vectors, which means that our random values look similar to preexisting embeddings

In [ ]:
vocab_rand, matrix_rand, dropped = random_rows_table(all_addr, min_count=2, seed=0)
n_w2v = len(vocab_mc2) - 2
print(f"word2vec rows: {n_w2v}, random rows: {len(dropped)}, matrix: {tuple(matrix_rand.shape)}")
print("dropped, all-digit:", sum(t.isdigit() for t in dropped), "| sample:", dropped[:5], dropped[-5:])

word2vec rows: 4282, random rows: 1868, matrix: (6152, 200)
dropped, all-digit: 699 | sample: ['004', '01', '02', '03', '06033'] ['YAWBUX', 'YOUNG', 'YOUNGS', 'ZLOTNICK', 'ZUMBACH']


### Double Check Random Row Formulation
1) no `UNK` left in any value for all_addr
2) rare number example doesn't occur i.e. when running on the same scenario, it doesn't show something like `UNK MAIN ST | UNK MAIN ST`
3) Assert that for two of random dropped values, which didn't pass the threshold, are both different and sufficiently disimilar 
4) Since random matrix is just normal matrix, with additional random matrices added for `UNK` indicies, verify that's actually the case, by checking equal for `matrix_rand[2 + n_w2v:]`

In [6]:
# no UNK left in any split
assert all(t in vocab_rand for s in all_addr for t in tokenize(s)), "some token still maps to UNK"

# same string -> same row, on both sides of an identical pair with a dropped number
grp = rare_number_group(val)
row = val[grp & (val["address_a"] == val["address_b"]).values].iloc[0]
ids_a, _ = encode_batch([row["address_a"]], vocab_rand)
ids_b, _ = encode_batch([row["address_b"]], vocab_rand)
assert torch.equal(ids_a, ids_b) and ids_a[0, 0] != UNK_ID and ids_a[0, 0] >= 2 + n_w2v, "rare number not on its own random row"
print("identical pair:", row["address_a"], "->", ids_a.tolist())

# different dropped tokens -> different rows
nums = [t for t in dropped if t.isdigit()][:2]
r0, r1 = matrix_rand[vocab_rand[nums[0]]], matrix_rand[vocab_rand[nums[1]]]
assert not torch.allclose(r0, r1)
print(f"{nums[0]} vs {nums[1]}: cos {torch.cosine_similarity(r0, r1, dim=0):.2f}")

# word2vec rows = 02's table; random rows on the same scale
assert torch.equal(matrix_rand[: 2 + n_w2v], matrix_mc2)
print("row norm, word2vec mean:", matrix_mc2[2:].norm(dim=1).mean().item(), "| random mean:", matrix_rand[2 + n_w2v:].norm(dim=1).mean().item())

identical pair: 4536 CONGRESS ST -> [[4924, 866, 2]]
004 vs 01: cos 0.27
row norm, word2vec mean: 2.8631844520568848 | random mean: 3.1257388591766357


### 3. Option 4: stacked `[word2vec mc1; fastText mc1]`
- We basically run `hstack` so that our embeddings now get `[w2v_vector, ft_vectors]`, and that's all we need

In [7]:
vocab_stack, matrix_stack = stacked_table(min_count=1)
print(f"rows: {len(vocab_stack)}, matrix: {tuple(matrix_stack.shape)}")

rows: 6151, matrix: (6152, 400)


- We will be running on `mc1`, so we're just checking to make sure `mc1` has no corresponding gaps
- Also verifying that the stacked embedding is the same as the `word2vec` embedding stacked on top of the `fast-text` embedding

In [8]:
from gensim.models import FastText, Word2Vec
w2v1 = Word2Vec.load("checkpoints/word2vec_50_mc1.model").wv
ft1 = FastText.load("checkpoints/fasttext_50_mc1.model").wv

assert all(t in vocab_stack for s in all_addr for t in tokenize(s)), "some token still maps to UNK"
for w in ["MAIN", "3015", "ST"]:
    expected = torch.cat([torch.tensor(w2v1[w]), torch.tensor(ft1[w])])
    assert torch.allclose(matrix_stack[vocab_stack[w]], expected), w
print("stacked rows = [word2vec; fastText] for MAIN, 3015, ST")

stacked rows = [word2vec; fastText] for MAIN, 3015, ST


A verbatim test to just ensure that the initial embeddings are FROZEN and that the UNK embedding is NOT FROZEN

In [9]:
small_train = PairDataset(train.iloc[:4 * BATCH_SIZE], train_label.iloc[:4 * BATCH_SIZE])   # 4 batches is enough
for name, vocab, matrix in [("rand_mc2", vocab_rand, matrix_rand), ("stack_mc1", vocab_stack, matrix_stack)]:
    torch.manual_seed(0)
    model = ESIM(matrix)
    table_before = model.embedding.frozen.weight.clone()
    head_before = model.head.linear2.weight.detach().clone()
    loader = DataLoader(small_train, batch_size=BATCH_SIZE, collate_fn=make_collate(vocab))
    train_epoch(model, loader, torch.optim.Adam(model.parameters(), lr=1e-3), nn.BCEWithLogitsLoss())
    assert torch.equal(model.embedding.frozen.weight, table_before), f"{name}: table changed"
    assert not torch.equal(model.head.linear2.weight, head_before), f"{name}: nothing trained"
    print(name, "table frozen, head trained")

rand_mc2 table frozen, head trained
stack_mc1 table frozen, head trained


## 4. Training
Same protocol as 02 (3 seeds, `is_better`, patience 3). Existing runs are skipped, so re-running the notebook only retrains what is missing. About 3 min per epoch per seed for option 5; option 4's first BiLSTM is wider.

In [8]:
tables = {
    "rand_mc2":  (vocab_rand, matrix_rand, {"table": "word2vec_50_mc2 + random rows (seed 0)"}),
    "stack_mc1": (vocab_stack, matrix_stack, {"table": "[word2vec_50_mc1; fasttext_50_mc1]"}),
}
for name, (vocab, matrix, config) in tables.items():
    folder = SETTINGS[name]
    loaders = make_loaders(vocab, splits)
    for seed in SEEDS:
        if os.path.exists(os.path.join(folder, f"esim_seed_{seed}.pt")):
            continue
        print(f"{'='*20} {name} seed {seed} {'='*20}")
        history, best, best_state = run_training(matrix, loaders, val, val_label, seed)
        save_run(folder, seed, history, best, best_state, vocab, matrix, loaders,
                 {**config, "emb_dim": matrix.shape[1], "dataset": DATASET})
        print(f"best: {best}\nproblems: {broken_run_problems(history)}")

==================== rand_mc2 seed 0 ====================


{'epoch': 0, 'train_loss': 0.061687865594416226, 'val_loss': 0.05681144446134567, 'found': 77.03889585947303, 'case1': 98.00449894782672, 'easy': 99.96517044712438}


{'epoch': 1, 'train_loss': 0.032494344182796814, 'val_loss': 0.039999645203351974, 'found': 77.79171894604768, 'case1': 99.04941586241927, 'easy': 99.9825852235622}


{'epoch': 2, 'train_loss': 0.028060489528829884, 'val_loss': 0.03421390801668167, 'found': 74.65495608531995, 'case1': 99.57187431971555, 'easy': 99.99564630589055}


{'epoch': 3, 'train_loss': 0.02567653067663967, 'val_loss': 0.056965503841638565, 'found': 77.66624843161857, 'case1': 98.57775197736014, 'easy': 99.9825852235622}


{'epoch': 4, 'train_loss': 0.024380695987184117, 'val_loss': 0.04577454552054405, 'found': 77.66624843161857, 'case1': 98.87526304332052, 'easy': 99.9825852235622}


... (56 outputs left out)

{'epoch': 4, 'train_loss': 0.02124303771433426, 'val_loss': 0.04087398201227188, 'found': 78.0426599749059, 'case1': 99.27436325375517, 'easy': 99.9825852235622}


{'epoch': 5, 'train_loss': 0.020506868597605293, 'val_loss': 0.032077111303806305, 'found': 78.16813048933501, 'case1': 99.45577244031638, 'easy': 99.9825852235622}


{'epoch': 6, 'train_loss': 0.019143754572118837, 'val_loss': 0.02942238561809063, 'found': 76.03513174404016, 'case1': 99.51382338001596, 'easy': 99.97823152945274}


best: {'epoch': 3, 'train_loss': 0.02333011819234134, 'val_loss': 0.03403344377875328, 'found': 78.92095357590966, 'case1': 99.02039039256948, 'easy': 99.99129261178109}
problems: []


## 5. Results: all five settings
- Check for any health problems, which is the same as `broken_run_problems`

In [9]:
# health: epochs trained and broken_run_problems per run ([] = healthy)
health()

epochs problems
setting   seed                 
mc2       0         11       []
          1          9       []
          2          8       []
mc1       0          5       []
          1         12       []
          2          8       []
ft_mc1    0          5       []
          1          8       []
          2          8       []
rand_mc2  0          5       []
          1         20       []
          2          8       []
stack_mc1 0         10       []
          1          9       []
          2          7       []

#### Results
Running basic checks on results of all the current splits now
- `rand_mc2` was chosen because it was the only one that passed the criteria, which was that it could handle the `UNK` number problem.
- There are some distributions that seem like they could indicate something, but within 1-2%, and a sample size of 3, the standard error is indistinguishable from noise.


In [10]:
# % correct per slice, one column per (setting, seed)
for name in ["val", "test"]:
    print(f"{'='*20} {name} {'='*20}")
    display(slice_table(*splits[name], name))

==================== val ====================


mc2                    mc1                  ft_mc1                 rand_mc2                 stack_mc1                
                                              0      1       2       0       1       2       0       1       2        0       1       2         0       1       2
slice                                                                                                                                                            
positives: identical                      98.6%  97.9%  100.0%   99.8%   98.8%  100.0%  100.0%  100.0%  100.0%   100.0%  100.0%  100.0%    100.0%  100.0%  100.0%
positives: non-identical                  82.7%  81.8%   80.9%   78.8%   80.3%   81.1%   79.9%   80.2%   81.6%    77.8%   81.8%   80.3%     80.8%   79.3%   78.9%
neg: case1 (same street, diff number)     98.3%  99.1%   98.2%   98.3%   98.7%   98.1%   98.4%   98.9%   98.2%    99.0%   99.0%   99.2%     98.6%   99.4%   99.0%
neg: case2 (same number, similar street)  55.9%  55.9%   55.9%   44.1%   52.9%   50.0%   50.0%   50.0%   50.0%    55.9%   55.9%   55.9%     50.0%   50.0%   58.8%
neg: case3 (easy)                         99.9%  99.9%  100.0%  100.0%  100.0%  100.0%  100.0%  100.0%  100.0%   100.0%  100.0%  100.0%     99.9%  100.0%  100.0%
neg: case2s (swapped look-alike street)   76.1%  79.8%   80.0%   72.7%   83.6%   74.4%   64.3%   71.0%   67.0%    74.6%   76.0%   82.2%     78.3%   79.6%   84.5%
overall (accuracy)                        98.4%  98.7%   98.5%   98.4%   98.7%   98.4%   98.3%   98.6%   98.3%    98.7%   98.8%   98.9%     98.6%   98.9%   98.9%

==================== test ====================


mc2                   mc1                 ft_mc1                 rand_mc2                stack_mc1                
                                              0      1       2      0      1       2       0       1       2        0       1      2         0       1       2
slice                                                                                                                                                         
positives: identical                      95.9%  95.5%  100.0%  99.5%  96.8%  100.0%  100.0%  100.0%  100.0%    99.9%  100.0%  99.9%    100.0%  100.0%  100.0%
positives: non-identical                  80.7%  77.9%   78.8%  76.6%  78.4%   78.4%   77.3%   76.9%   78.4%    74.9%   78.9%  78.2%     78.4%   76.7%   75.3%
neg: case1 (same street, diff number)     98.3%  98.4%   96.1%  97.3%  98.3%   95.7%   98.5%   99.0%   97.7%    98.9%   98.9%  98.6%     97.6%   99.6%   98.0%
neg: case2 (same number, similar street)  40.0%  46.7%   40.0%  26.7%  26.7%   33.3%   33.3%   60.0%   46.7%    33.3%   26.7%  46.7%     46.7%   46.7%   46.7%
neg: case3 (easy)                         99.9%  99.9%   99.9%  99.9%  99.9%   99.9%  100.0%  100.0%   99.9%   100.0%   99.9%  99.9%     99.9%  100.0%   99.9%
neg: case2s (swapped look-alike street)   74.2%  75.5%   73.9%  72.1%  79.6%   74.5%   68.9%   72.1%   70.5%    72.9%   75.9%  74.1%     80.2%   77.8%   80.7%
overall (accuracy)                        98.0%  98.1%   97.6%  97.9%  98.2%   97.5%   98.3%   98.5%   98.1%    98.4%   98.6%  98.4%     98.3%   98.8%   98.3%

### Comparing Rare Numbers
- When isolating for `UNK` numbers, `rand_mc2` is the only one that passed the threshold of `145/161` matches for `case1`

In [11]:
# rare-number group (02's D2-D5): identical positives found / case1 negatives rejected inside it, errors outside it
rare_number_table({"val": splits["val"], "test": splits["test"]})

identical UNK: found case1 UNK: rejected  identical other: missed  case1 other: errors
split setting   seed                                                                                       
val   mc2       0                 29 / 78           141 / 161                        0                  216
                1                  9 / 78           158 / 161                        1                  124
                2                 78 / 78             0 / 161                        0                   92
      mc1       0                 72 / 78            21 / 161                        0                   98
                1                 38 / 78           111 / 161                        1                  125
                2                 78 / 78            12 / 161                        0                  116
      ft_mc1    0                 78 / 78           151 / 161                        0                  213
                1                 78 / 78           151 / 161                        0                  137
                2                 78 / 78           144 / 161                        0                  229
      rand_mc2  0                 78 / 78           158 / 161                        0                  128
                1                 78 / 78           155 / 161                        0                  132
                2                 78 / 78           154 / 161                        0                  100
      stack_mc1 0                 78 / 78           126 / 161                        0                  155
                1                 78 / 78           148 / 161                        0                   72
                2                 78 / 78           128 / 161                        0                  102
test  mc2       0                32 / 178           381 / 404                        0                  225
                1                20 / 178           387 / 404                        0                  207
                2               178 / 178             0 / 404                        1                  162
      mc1       0               162 / 178           149 / 404                        1                  129
                1                72 / 178           316 / 404                        7                  160
                2               178 / 178            64 / 404                        0                  276
      ft_mc1    0               178 / 178           376 / 404                        0                  194
                1               178 / 178           388 / 404                        0                  124
                2               178 / 178           350 / 404                        0                  278
      rand_mc2  0               178 / 178           385 / 404                        3                  143
                1               178 / 178           392 / 404                        0                  152
                2               178 / 178           379 / 404                        2                  171
      stack_mc1 0               178 / 178           291 / 404                        0                  232
                1               178 / 178           386 / 404                        0                   40
                2               178 / 178           310 / 404                        0                  202

**Result (David, 2026-11-08; written down by Claude from the discussion):**

**Decision rule, fixed before reading val:**

| slice | role | why |
|---|---|---|
| identical positives | ignored | an exact-string pre-check handles them before the model |
| case1, rare-number group | must-pass: ≥ 145 / 161 rejected (≥ 90%) on every val seed | reading house numbers is the minimum; the pre-check does not remove these pairs, they differ as strings |
| non-identical positives | decides | |
| `case2s` | decides (hardest slice) | |
| case2 (real, n = 34 val) | ignored | too small, seed noise |
| case3 (easy) | sanity check only | |

A gap counts only if it is bigger than the seed spread (min–max over 3 seeds). Note: non-identical and `case2s` pull in opposite directions (a stricter model gains `case2s` and loses rewrites), so a model that only leans toward "no" wins one and loses the other.

**Must-pass (val, `case1 UNK: rejected`):** `mc2` 141 / 158 / 0, `mc1` 21 / 111 / 12, `ft_mc1` 151 / 151 / 144, `rand_mc2` 158 / 155 / 154, `stack_mc1` 126 / 148 / 128. **Only `rand_mc2` passes** (`ft_mc1` misses by one pair on seed 2; the cut was not moved).

**Why random rows work:** the group's question is identity ("same number?"), not meaning. A fixed random row per dropped token gives the same vector for the same token and a near-orthogonal one for a different token, so alignment finds an exact partner or none. word2vec places same-street numbers close (cos 0.92), fastText fairly close (0.79), and stacking the two still says "close".

**Cost vs `mc2` on the deciding slices (val, seeds 0 / 1 / 2):** non-identical 82.7 / 81.8 / 80.9 vs 77.8 / 81.8 / 80.3; `case2s` 76.1 / 79.8 / 80.0 vs 74.6 / 76.0 / 82.2. Ranges overlap on both. **Conclusion: `rand_mc2` fixes the rare-number group; no difference on non-identical or `case2s` is detectable at 3 seeds, a cost of about 1 point is not ruled out** (more seeds would settle it; val's ~800–900 rows per slice also limit resolution to about ±1.4 pts).

**`results.md`:** `mc2` (the reproduction) and `rand_mc2` (the fix) as test min–max (val min–max); the other three settings get one sentence each in the Step 4 paragraph.


## Diagnostics
One-off inspections, kept out of the pipeline cells above (question → result → query).

**Swap test (`rand_mc2`, val).** 
- Check if we have any preexisting reliance on ordering, regarding address side
- Results show flipped predictions of `2.3–4.3%` for `non-identical` and `2.3–4.1%` for `case2s`. case1 `0.1–0.5%`, easy ≈ 0, overall `0.15–0.34%`. Largest probability change 0.36–0.56: the flips are pairs near the threshold. Not a huge amount, but indicates some learned behavior

In [10]:
swapped = val.rename(columns={"address_a": "address_b", "address_b": "address_a"})
loader_ab = make_loaders(vocab_rand, {"val": (val, val_label)})["val"]
loader_ba = make_loaders(vocab_rand, {"val": (swapped, val_label)})["val"]

rows = []
for seed in SEEDS:
    ckpt = torch.load(f"{SETTINGS['rand_mc2']}/esim_seed_{seed}.pt")
    model = ESIM(matrix_rand)
    model.load_state_dict(ckpt["state_dict"])
    p_ab = predict(model, loader_ab)            # probabilities, (a, b)
    p_ba = predict(model, loader_ba)            # probabilities, (b, a)
    ab = (p_ab >= THRESHOLD).int().numpy()
    ba = (p_ba >= THRESHOLD).int().numpy()
    assert np.array_equal(ab, np.load(f"{SETTINGS['rand_mc2']}/preds_val_seed_{seed}.npy")), "unswapped run != saved predictions"
    flips = ab != ba
    rows.append({"seed": seed, "flips": int(flips.sum()), "of": len(flips), "% flipped": 100 * flips.mean(),
                 "max |p_ab - p_ba|": (p_ab - p_ba).abs().max().item(),
                 **{f"flips {t}": int(flips[(val["neg_type"] == t).values].sum()) for t in sorted(val["neg_type"].unique())}})
pd.DataFrame(rows).set_index("seed")


,flips,of,% flipped,max |p_ab - p_ba|,flips case0,flips case1,flips case2,flips case2s,flips case3
seed,,,,,,,,,
0,68,41865,0.162427,0.543069,25,14,0,26,3
1,144,41865,0.343963,0.564962,34,67,0,37,6
2,63,41865,0.150484,0.358619,18,21,0,21,3


**Symmetric prediction (`rand_mc2`, val).** 
- We just check if running a version which averages both symmetric representations, provides a meaningful improvement to scores.
- Results show that there are very few results


In [11]:
from helpers.check_model_performance import review_typical_slices

class SymmetricESIM(nn.Module):
    """Wraps a trained ESIM; logit = mean of both input orders."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, ids_a, len_a, ids_b, len_b):
        return (self.model(ids_a, len_a, ids_b, len_b) + self.model(ids_b, len_b, ids_a, len_a)) / 2

# test on one batch: symmetric output is identical for swapped inputs
ids_a, len_a, ids_b, len_b, _ = next(iter(loader_ab))
ckpt = torch.load(f"{SETTINGS['rand_mc2']}/esim_seed_0.pt")
m = ESIM(matrix_rand); m.load_state_dict(ckpt["state_dict"]); m.eval()
sym = SymmetricESIM(m)
with torch.no_grad():
    assert torch.allclose(sym(ids_a, len_a, ids_b, len_b), sym(ids_b, len_b, ids_a, len_a))
print("symmetric on swapped inputs: OK")

cols, changed = {}, {}
for seed in SEEDS:
    ckpt = torch.load(f"{SETTINGS['rand_mc2']}/esim_seed_{seed}.pt")
    model = ESIM(matrix_rand)
    model.load_state_dict(ckpt["state_dict"])
    plain = np.load(f"{SETTINGS['rand_mc2']}/preds_val_seed_{seed}.npy")
    symm = (predict(SymmetricESIM(model), loader_ab) >= THRESHOLD).int().numpy()
    changed[seed] = int((plain != symm).sum())
    for name, preds in [("plain", plain), ("symmetric", symm)]:
        cols[(seed, name)] = review_typical_slices(val, preds, val_label).set_index("slice")["% correct"]
print("predictions changed by averaging, per seed:", changed)
pd.DataFrame(cols)


symmetric on swapped inputs: OK
predictions changed by averaging, per seed: {0: 36, 1: 63, 2: 26}


0                 1                 2          
                                           plain symmetric   plain symmetric   plain symmetric
slice                                                                                         
positives: identical                      100.0%    100.0%  100.0%    100.0%  100.0%    100.0%
positives: non-identical                   77.8%     77.9%   81.8%     82.2%   80.3%     79.7%
neg: case1 (same street, diff number)      99.0%     99.0%   99.0%     98.9%   99.2%     99.3%
neg: case2 (same number, similar street)   55.9%     55.9%   55.9%     55.9%   55.9%     55.9%
neg: case3 (easy)                         100.0%    100.0%  100.0%    100.0%  100.0%    100.0%
neg: case2s (swapped look-alike street)    74.6%     74.6%   76.0%     76.3%   82.2%     82.0%
overall (accuracy)                         98.7%     98.7%   98.8%     98.7%   98.9%     98.9%

**AUC, non-identical vs `case2s`, setting `mc2` (Claude, 2026-11-11, for `results.md`'s AUC column).** Probabilities saved as `probs_{split}_seed_{s}.npy` in `esim_mc2/` (`>= 0.5` reproduces the saved predictions). Test (val): 0.849 (0.867) / 0.845 (0.882) / 0.850 (0.883). `rand_mc2`'s AUC is in `transformers/01_bi_encoder.ipynb` D3–D4. Query below.


In [ ]:
from sklearn.metrics import roc_auc_score
vocab_mc2_t, matrix_mc2_t = plain_table("word2vec", 2)
loaders_mc2 = make_loaders(vocab_mc2_t, {"val": (val, val_label), "test": (test, test_label)})
for seed in SEEDS:
    m = ESIM(matrix_mc2_t)
    m.load_state_dict(torch.load(f"{SETTINGS['mc2']}/esim_seed_{seed}.pt", weights_only=False)["state_dict"])
    for split, (df, labels) in {"val": (val, val_label), "test": (test, test_label)}.items():
        path = f"{SETTINGS['mc2']}/probs_{split}_seed_{seed}.npy"
        if not os.path.exists(path):
            np.save(path, predict(m, loaders_mc2[split]).numpy())
        p = np.load(path)
        pos = (labels.values == 1) & (df["address_a"] != df["address_b"]).values
        c2s = (df["neg_type"] == "case2s").values
        print(seed, split, round(roc_auc_score(np.r_[np.ones(pos.sum()), np.zeros(c2s.sum())], np.r_[p[pos], p[c2s]]), 3))
